In [ ]:
import sys
sys.path.insert(0,"../../../")

import joblib
import pandas as pd
import json
import os
from typing import List, Dict, Tuple, Set
from copy import deepcopy
import ast

from src.utils.data_structs import NodeType, RelationType, Triplet, create_id
from collections import defaultdict


In [ ]:
BASE_DIR = "../../.." # TO CHANGE
SELECTED_BENCHMARK_NAME = "hotpotqa" # TO CHANGE
SELECTED_KNOWLEDGEGRAPH_NAME = "qwen257b_230126_v2prompts" # TO CHANGE

QA_DATASETS_DIR = f"{BASE_DIR}/data/qa_datasets"
KNOWLEDGE_GRAPHS_DIR = f"{BASE_DIR}/../pai2_knowledge_graphs"
EXTRACTED_TRIPLES_DIRNAME = 'tmp_extracted_triplets'
TRIPLES_NODES_IDS_FILENAME = 'partial_triples_nodes_ids.joblib'
MATCHED_TNIDS_TO_QUESTIONS_FILENAME = 'matched_tnids_to_questions.json'

BENCHMARK_PATH = {
    'nq': {
        'qa': f"{QA_DATASETS_DIR}/natural_questions_train/qa_pairs.csv",
        'documents': f"{QA_DATASETS_DIR}/natural_questions_train/relevant_contexts.csv"
    }, 
    'triviaqa': {
        'qa': f"{QA_DATASETS_DIR}/v2/trivia_qa_rcwikipedia_validation/qa_pairs.csv",
        'documents': f"{QA_DATASETS_DIR}/v2/trivia_qa_rcwikipedia_validation/relevant_contexts.csv"
    }, 
    'hotpotqa': {
        'qa': f"{QA_DATASETS_DIR}/v2/hotpotqa_distractor_validation/qa_pairs.csv",
        'documents': f"{QA_DATASETS_DIR}/v2/hotpotqa_distractor_validation/relevant_contexts.csv"
    }, 
    '2wiki': {
        'qa': f"{QA_DATASETS_DIR}/2wikimultihopqa_dev/qa_pairs.csv",
        'documents': f"{QA_DATASETS_DIR}/2wikimultihopqa_dev/relevant_contexts.csv"
    }, 
    'musique': {
        'qa': f"{QA_DATASETS_DIR}/musique_validation/qa_pairs.csv",
        'documents': f"{QA_DATASETS_DIR}/musique_validation/relevant_contexts.csv"
    }
}

EXTRACTED_TRIPLES_PATH = f"{KNOWLEDGE_GRAPHS_DIR}/{SELECTED_BENCHMARK_NAME}/{SELECTED_KNOWLEDGEGRAPH_NAME}/{EXTRACTED_TRIPLES_DIRNAME}"
SAVE_TRIPLES_NODES_IDS_PATH = f"{KNOWLEDGE_GRAPHS_DIR}/{SELECTED_BENCHMARK_NAME}/{SELECTED_KNOWLEDGEGRAPH_NAME}/{TRIPLES_NODES_IDS_FILENAME}"
SAVE_MATCHED_TNIDS_TO_QUESTIONS_PATH = f"{KNOWLEDGE_GRAPHS_DIR}/{SELECTED_BENCHMARK_NAME}/{SELECTED_KNOWLEDGEGRAPH_NAME}/{MATCHED_TNIDS_TO_QUESTIONS_FILENAME}"

In [ ]:
EXTRACTED_TRIPLES = dict()
extracted_t_files = os.listdir(EXTRACTED_TRIPLES_PATH)
for t_file in extracted_t_files:
    triples_idx = int(t_file[4:])
    EXTRACTED_TRIPLES[triples_idx] = joblib.load(f"{EXTRACTED_TRIPLES_PATH}/{t_file}")

In [ ]:
print(len(EXTRACTED_TRIPLES))

In [ ]:
len(EXTRACTED_TRIPLES[0])

In [ ]:
BENCHMARK_DOCUMENTS = pd.read_csv(BENCHMARK_PATH[SELECTED_BENCHMARK_NAME]['documents'], sep=',')
BENCHMARK_QA = pd.read_csv(BENCHMARK_PATH[SELECTED_BENCHMARK_NAME]['qa'], sep=',')
print(BENCHMARK_DOCUMENTS.shape)
print(BENCHMARK_QA.shape)

In [ ]:
BENCHMARK_DOCUMENTS.head()

In [ ]:
#BENCHMARK_DOCUMENTS['context'][2362]

In [ ]:
BENCHMARK_QA.head()

In [ ]:
EXTRACTED_TRIPLES_NODES_IDS: Dict = dict()
MATCHED_TNIDS_TO_QUESTIONS: Dict = dict()

In [ ]:
#EXTRACTED_TRIPLES[2362]

In [ ]:
for idx in range(BENCHMARK_DOCUMENTS.shape[0]):
    #cur_cntx_idx = BENCHMARK_DOCUMENTS['cntx_idx'][idx]
    cur_triples: List[Triplet] = EXTRACTED_TRIPLES[idx]

    uniques_object_nodes_info: Set[Tuple[str,str]] = set()
    uniques_thesis_nodes_info: Set[Tuple[str,str]] = set()
    unique_triples_ids: Dict[str, Set[str]] = defaultdict(set)
    tripletid_to_thesisid: Dict[str, str] = dict()
    for cur_triplet in cur_triples:
        if cur_triplet.start_node.type == NodeType.object:
            uniques_object_nodes_info.add((cur_triplet.start_node.id, cur_triplet.start_node.name))
        elif cur_triplet.start_node.type == NodeType.hyper:
            uniques_thesis_nodes_info.add((cur_triplet.start_node.id, cur_triplet.start_node.name))
            tripletid_to_thesisid[cur_triplet.id] = cur_triplet.start_node.id
        
        if cur_triplet.end_node.type == NodeType.object:
            uniques_object_nodes_info.add((cur_triplet.end_node.id, cur_triplet.end_node.name))
        elif cur_triplet.end_node.type == NodeType.hyper:
            uniques_thesis_nodes_info.add((cur_triplet.end_node.id, cur_triplet.end_node.name))
            tripletid_to_thesisid[cur_triplet.id] = cur_triplet.end_node.id
                    
        unique_triples_ids[cur_triplet.relation.type.value].add(cur_triplet.id)

    EXTRACTED_TRIPLES_NODES_IDS[int(idx)] = {
        'unique_object_nodes_info': deepcopy(uniques_object_nodes_info),
        'uniques_thesis_nodes_info': deepcopy(uniques_thesis_nodes_info),
        'unique_triples_ids': deepcopy(dict(unique_triples_ids)),
        'tripletid_to_thesisid': deepcopy(tripletid_to_thesisid)
    }
EXTRACTED_TRIPLES_NODES_IDS = dict(EXTRACTED_TRIPLES_NODES_IDS)

In [ ]:
len(EXTRACTED_TRIPLES_NODES_IDS)

In [ ]:
#EXTRACTED_TRIPLES_NODES_IDS[2362]

In [ ]:
for idx in range(BENCHMARK_QA.shape[0]):
    qa_idx = BENCHMARK_QA['qa_idx'][idx]
    relevant_cntx_idxs = ast.literal_eval(BENCHMARK_QA['cntx_ids'][idx])

    uniques_object_nodes_info: Set[Tuple[str,str]] = set()
    uniques_thesis_nodes_info: Set[Tuple[str,str]] = set()
    unique_triples_ids: Dict[str, Set[str]] = defaultdict(set)
    tripletid_to_thesisid: Dict[str, str] = dict()

    for cur_cntx_idx in relevant_cntx_idxs:
        uniques_object_nodes_info.update(EXTRACTED_TRIPLES_NODES_IDS[cur_cntx_idx]['unique_object_nodes_info'])
        uniques_thesis_nodes_info.update(EXTRACTED_TRIPLES_NODES_IDS[cur_cntx_idx]['uniques_thesis_nodes_info'])
        tripletid_to_thesisid.update(EXTRACTED_TRIPLES_NODES_IDS[cur_cntx_idx]['tripletid_to_thesisid'])
        for k, v in EXTRACTED_TRIPLES_NODES_IDS[cur_cntx_idx]['unique_triples_ids'].items():
            unique_triples_ids[k].update(v)

    question_hash = create_id(BENCHMARK_QA['question'][idx])
    MATCHED_TNIDS_TO_QUESTIONS[question_hash] = {
        'qa_idx': int(qa_idx),
        'question': BENCHMARK_QA['question'][idx],
        'cntx_idxs': deepcopy(relevant_cntx_idxs),
        'relevant_unique_object_nodes_info': deepcopy({item[1]: item[0] for item in list(uniques_object_nodes_info)}),
        'relevant_unique_thesis_nodes_info': deepcopy({item[0]: item[1] for item in list(uniques_thesis_nodes_info)}),
        'relevant_tripletid_to_thesisid': deepcopy(tripletid_to_thesisid),
        'relevant_unique_triples_ids': deepcopy({k: list(v) for k,v in dict(unique_triples_ids).items()})
    }

In [ ]:
print(len(MATCHED_TNIDS_TO_QUESTIONS))

In [ ]:
#MATCHED_TNIDS_TO_QUESTIONS['5d630ab3267d560b99884eb2530a6463']

In [ ]:
joblib.dump(EXTRACTED_TRIPLES_NODES_IDS, SAVE_TRIPLES_NODES_IDS_PATH)

In [ ]:
with open(SAVE_MATCHED_TNIDS_TO_QUESTIONS_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(MATCHED_TNIDS_TO_QUESTIONS, indent=1, ensure_ascii=False))